# Notebook 09 — Ablation Study

> **阶段**：Stage 5 Research · **预计时间**：30 分钟设计 + GPU 实验时间 · **平台**：Kaggle Notebook

设计你的第一个 Controlled Experiment：一次只改变一个主要变量。


# Learning Objectives

- 设计四组消融：Prompt / 训练数据量 / LoRA rank / 图像分辨率；
- 坚持唯一变量原则并记录全部固定量；
- 生成 ablation_results.csv 与图表；
- 回答：哪个变量最影响 Text / Table / Formula？是否存在 diminishing returns？


# Why This Matters

消融是「变量 → 结果」因果推断的最低门槛。没有消融的提升报告，无法排除「分数涨了只是因为换了个 prompt」。


# Concepts

| Ablation | 变量 | 取值（Kaggle 可调） |
| --- | --- | --- |
| A Prompt | prompt_id | v0（simple）vs v2/v3（structured） |
| B 数据量 | n_train | 25 / 50 / 100 / 200（GPU；CPU 冒烟 2/4/8） |
| C LoRA Rank | r | 4 / 8 / 16（其余超参固定） |
| D 分辨率 | image size | low / medium / high（按 processor 支持设置） |

每组实验的固定量：模型 revision、数据划分 seed、评测 commit、generation config、评测页面集合。


## Step 1 — Ablation A（Prompt，复用 Notebook 03 结果）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
from src import data
from src.config import project_root

bench = project_root() / 'results' / 'prompt_benchmark.json'
if bench.is_file():
    print(json.dumps(data.read_json(bench), ensure_ascii=False, indent=2))
else:
    print('先运行 Notebook 03 生成 prompt_benchmark.json。')


## Step 2 — Ablation B/C/D 的运行骨架

下面的函数是骨架：B 复用 Notebook 05 的 train_sft，C 复用 Notebook 06 的 setup_lora，D 在预处理图像时改变分辨率。CPU 上只做最小冒烟；真实消融在 GPU 上跑并回填 ablation_results.csv。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.ablation import ablation_record

FIXED = {'model_revision': 'ce51f56c', 'dataset_revision': 'aa1ee96d',
        'eval_commit': '193627ae', 'seed': 42, 'eval_pages': 'fast-12'}

# TODO: 每个取值跑完训练 + Notebook 07 评测后，把官方指标填入 metrics。
# 示例（数据量消融）：
rows_demo = [
    ablation_record('B_data_size', 'n_train', '25', {'text': None, 'table': None, 'formula': None, 'overall': None}, FIXED),
    ablation_record('B_data_size', 'n_train', '50', {'text': None, 'table': None, 'formula': None, 'overall': None}, FIXED),
    ablation_record('B_data_size', 'n_train', '100', {'text': None, 'table': None, 'formula': None, 'overall': None}, FIXED),
    ablation_record('B_data_size', 'n_train', '200', {'text': None, 'table': None, 'formula': None, 'overall': None}, FIXED),
]
print(rows_demo[0])


## Step 3 — 汇总 ablation_results.csv 并绘图


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.ablation import write_ablation_csv, plot_ablation
from src.config import project_root

csv_path = write_ablation_csv(rows_demo, project_root() / 'results' / 'ablation_results.csv')
print('CSV:', csv_path)
# fig = plot_ablation(rows_demo, x_key='n_train', y_keys=['text', 'table', 'formula', 'overall'], title='Ablation B: data size')
# display(fig)


# What You Should Observe

- 如果某个变量的曲线先升后平，就是 diminishing returns；
- Text/Table/Formula 对同一变量的响应可能不同——这正是研究问题的来源；
- 任何一行缺失固定量记录，该行都不可信。


# Research Checkpoint

> 用（实测或预期的）曲线回答：哪个变量最影响 Text？哪个最影响 Table/Formula？哪个增加算力但几乎没有提升？是否存在 diminishing returns？

**TODO：** 答案写入 `results/nb09/research_checkpoint.md`。


# Exercises

1. **TODO：** 在 GPU 上完成 Ablation B（至少 2 个数据量取值），回填 CSV；
2. **TODO：** 完成 Ablation C（r=4/8/16），记录 VRAM/时间，与参数量对照；
3. **TODO：** 设计 Ablation D 的分辨率档位（查 SmolDocling processor 支持的输入尺寸），并说明分辨率如何影响推理成本与表格/公式识别。


# Takeaways

- 消融的价值在于「可归因」，不在「图多」；
- 固定量记录与唯一变量同样重要；
- diminishing returns 是停止加预算的科学理由。

**下一步**：[Notebook 10](10_From_Experiments_to_Research_Questions.ipynb) — 从实验走向科研问题。
